In [0]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pyspark.sql import SparkSession

In [0]:


spark = SparkSession.builder.appName("Financial_Dashboard").getOrCreate()

# COMMAND ----------
# MAGIC %md
# MAGIC # 1. Ingestão de Dados (The Lakehouse)
# COMMAND ----------

try:
    # 1. Carrega as Previsões Matemáticas geradas pelo Script 2 no Banco de Dados
    df_pred_spark = spark.table("foodshop_predictions")
    df_pred = df_pred_spark.toPandas()
    df_pred['date'] = pd.to_datetime(df_pred['date'])
    
    # Criar a safra MENSAL (Ex: '2026-04') porque Preços de Contrato Livres são negociados ao mês!
    df_pred['month_num'] = df_pred['date'].dt.strftime('%Y-%m')
    
    # Empilhar/Agregar o Volume Previsto MENSALMENTE
    df_month = df_pred.groupby('month_num', as_index=False)['predicted_mwh'].sum()
    df_month.rename(columns={'predicted_mwh': 'predicted_mwh_consumption'}, inplace=True)
    
except Exception as e:
    print("CRITICAL ERROR: Table 'foodshop_predictions' not found!")
    print("Make sure to run the ML Module Script (Script 2) to generate the forecast first.")
    raise e

# MOCK DE ESTRESSE: Como o Consumo Real dos 90 dias do futuro estrito (2026+) ainda não existiram na vida real,
# não conseguimos saber o 'Desvio de Erro Absoluto' real do futuro. 
# Simularemos que o Desvio será o Consumo Previsto pela IA +/- 5% de margem de erro técnico orgânico.
# Na operação corporativa de longo prazo real, você importará a ONS cruzada aqui.
np.random.seed(42)
df_month['actual_mwh_consumption'] = df_month['predicted_mwh_consumption'] * np.random.uniform(0.95, 1.05, size=len(df_month))

# COMMAND ----------
# MAGIC %md
# MAGIC # 2. Preços Spot do Mercado (PLD)
# COMMAND ----------

# Criamos uma matriz flutuante artificial imitando exatamente o comportamento histórico que testou no Step 6.
proxy_dict = {
    'month_num': list(df_month['month_num'].values),
    
    # Imitação dos precos malucos do Step6 pro Trimestre: Ex [R$ 265, R$ 59, R$ 93]
    'spot_price': [265.00, 59.00, 93.00][:len(df_month)] if len(df_month) >= 3 else [180.00] * len(df_month) 
}

if len(df_month) > 3:
    proxy_dict['spot_price'] = proxy_dict['spot_price'] + [110.00] * (len(df_month) - 3)

df_pld_proxy = pd.DataFrame(proxy_dict)

# Merge Financeiro para criar a Matriz Geral do Dashboard Executivo
df_final = df_month.merge(df_pld_proxy[['month_num', 'spot_price']], on='month_num', how='left')

# COMMAND ----------
# MAGIC %md
# MAGIC # 3. Classificação Analítica de Risco (Controladoria)
# COMMAND ----------

# Constante de Negócio Padrão
FIXED_CONTRACT_PRICE = 160.00

def classify_risk(gap):
    """ Réplica precisa Matemática da Sessão 7 """
    if gap <= 0:
        return 'Level 0 (Green)', 'Spot Market', 'Do not lock. Buy at spot.'
    elif 0 < gap <= 0.15:
        return 'Level 1 (Yellow)', 'Thin Margin', 'Lock Contract (Thin Margin)'
    elif 0.15 < gap <= 0.50:
        return 'Level 2 (Orange)', 'Hedge Recommended', 'Lock Contract'
    else:
        return 'Level 3 (Red)', 'Critical Alert', 'LOCK CONTRACT NOW'

def prescriptive_engine(df, contract_price):
    df = df.copy()
    df['contract_price'] = contract_price
    df['gap_percentage'] = (df['spot_price'] - df['contract_price']) / df['contract_price']
    
    # Assinatura Funcional Global
    res = [classify_risk(g) for g in df['gap_percentage']]
    df[['risk_level', 'status', 'suggested_action']] = pd.DataFrame(res, index=df.index)
    df['gap_display'] = (df['gap_percentage'] * 100).round(2).astype(str) + '%'
    return df

# Rode o motor central gerando a grade de tomadas de decisões do Comitê
df_dashboard = prescriptive_engine(df_final, FIXED_CONTRACT_PRICE)

# COMMAND ----------
# MAGIC %md
# MAGIC # 4. Accounting e Fechamento Comercial
# COMMAND ----------

def calculate_strategic_cost(row):
    """
    Simula uma barreira de Hedge. 
    Aplica a punição por Erros de IA se o volume real diferir gravemente do Traçado a Fixo.
    """
    if 'Lock' in str(row['suggested_action']) or 'LOCK' in str(row['suggested_action']):
        cost = (row['predicted_mwh_consumption'] * FIXED_CONTRACT_PRICE) + \
               ((row['actual_mwh_consumption'] - row['predicted_mwh_consumption']) * row['spot_price'])
    else:
        # Pagar na cabeça o Preço Spot pra tudo no improviso
        cost = row['actual_mwh_consumption'] * row['spot_price']
    return cost

df_dashboard['ai_cost_brl'] = df_dashboard.apply(calculate_strategic_cost, axis=1)

# Custo Cego se a Empresa se recusasse a possuir ML e pagasse Spot para todas as alocações daquele mês
df_dashboard['base_cost_brl'] = df_dashboard['actual_mwh_consumption'] * df_dashboard['spot_price']

# Fechamento Financeiro 
df_dashboard['saving_brl'] = df_dashboard['base_cost_brl'] - df_dashboard['ai_cost_brl']
df_dashboard['efficiency_percentage'] = (df_dashboard['saving_brl'] / df_dashboard['base_cost_brl']) * 100

# Formatação Literal Bruta
df_dashboard['formatted_ai_cost'] = df_dashboard['ai_cost_brl'].apply(lambda x: f"R$ {x:,.2f}")
df_dashboard['formatted_base_cost'] = df_dashboard['base_cost_brl'].apply(lambda x: f"R$ {x:,.2f}")
df_dashboard['formatted_saving'] = df_dashboard['saving_brl'].apply(lambda x: f"R$ {x:,.2f}")

# COMMAND ----------
# MAGIC %md
# MAGIC # 5. Interface Executiva Front-End Dashboard
# COMMAND ----------

print("-" * 50)
print(f"<<<<<<<<<<< PREDICTIVE REPORT & RISK FINANCE >>>>>>>>>>>")
print(f"Estimated AI Generated Savings (Quarter): R$ {df_dashboard['saving_brl'].sum():,.2f}")
print(f"Immediate Strategy (Month 1): {df_dashboard['suggested_action'].iloc[0]}")
print("-" * 50)
print("\n")

raw_vision_columns = [
    'month_num', 'predicted_mwh_consumption', 'spot_price', 'contract_price', 
    'gap_display', 'risk_level', 'suggested_action', 
    'formatted_saving', 'efficiency_percentage'
]

df_view = df_dashboard[raw_vision_columns].copy()
df_view.columns = [
    'Harvest Cycle (Month)', 'Predicted MWh (AI)', 'Spot Market Cost', 'Fixed Derivative Rate', 
    'Margin Cost/Gain %', 'Alert Level', 'Commercial Buying Directive', 
    'Saved Operational Cash (R$)', 'ROI Return (%)'
]

def map_risk_styles(row):
    """ Engine de cores nativa que transita para os Dashboards SQL do sistema da Empresa """
    colors = [
        'background-color: #f8d7da; color: #721c24; font-weight: bold;' if 'Level 3' in str(v) else
        'background-color: #ffe5d0; color: #854000;' if 'Level 2' in str(v) else
        'background-color: #fff3cd; color: #856404;' if 'Level 1' in str(v) else
        'background-color: #d4edda; color: #155724;' if 'Level 0' in str(v) else '' 
        for v in row
    ]
    return colors

# Constrói o HTML Visível
visual_html_table = df_view.style.apply(map_risk_styles, axis=1)\
                            .format({
                                'Predicted MWh (AI)': '{:,.1f} MWh', 
                                'ROI Return (%)': '{:.2f}%', 
                                'Spot Market Cost': 'R$ {:.2f}', 
                                'Fixed Derivative Rate': 'R$ {:.2f}'
                            })\
                            .set_properties(**{'border': '1px solid black', 'text-align': 'center', 'padding': '5px'})\
                            .set_table_styles([dict(selector='th', props=[('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center')])])

# Renderização do Objeto Final Analítico - (Apenas CLIQUE em PIN TO DASHBOARD em cima da renderização dessa tabela abaixo no próprio Notebook)
display(visual_html_table)

# COMMAND ----------
# MAGIC %md
# MAGIC # 6. Financial Executive Summary (ROI comparison)
# COMMAND ----------

summary_columns = [
    'month_num', 'formatted_base_cost', 'formatted_ai_cost', 'formatted_saving', 'efficiency_percentage'
]

df_roi_summary = df_dashboard[summary_columns].copy()
df_roi_summary.columns = [
    'Month', 'Cost Without AI (Spot)', 'Cost With AI (Hedge)', 'Saving (R$)', 'Efficiency (%)'
]

# Estilo escuro nativo do Databricks igual ao print que você mandou
roi_styled = df_roi_summary.style\
    .format({'Efficiency (%)': '{:.2f}'})\
    .set_properties(**{'text-align': 'right', 'background-color': '#112233', 'color': '#A6E22E'})\
    .set_table_styles([dict(selector='th', props=[('background-color', '#0B141C'), ('color', '#F8F8F2'), ('text-align', 'right')])])\
    .hide(axis='index')

print("\n--- AI FINANCIAL ROI SUMMARY ---")
display(roi_styled)


--------------------------------------------------
<<<<<<<<<<< PREDICTIVE REPORT & RISK FINANCE >>>>>>>>>>>
Estimated AI Generated Savings (Quarter): R$ 227,380.65
Immediate Strategy (Month 1): LOCK CONTRACT NOW
--------------------------------------------------




,Harvest Cycle (Month),Predicted MWh (AI),Spot Market Cost,Fixed Derivative Rate,Margin Cost/Gain %,Alert Level,Commercial Buying Directive,Saved Operational Cash (R$),ROI Return (%)
0,2026-03,"2,165.5 MWh",R$ 265.00,R$ 160.00,65.62%,Level 3 (Red),LOCK CONTRACT NOW,"R$ 227,380.65",40.13%
1,2026-04,"15,845.1 MWh",R$ 59.00,R$ 160.00,-63.12%,Level 0 (Green),Do not lock. Buy at spot.,R$ 0.00,0.00%
2,2026-05,"15,302.9 MWh",R$ 93.00,R$ 160.00,-41.88%,Level 0 (Green),Do not lock. Buy at spot.,R$ 0.00,0.00%
3,2026-06,"12,427.5 MWh",R$ 110.00,R$ 160.00,-31.25%,Level 0 (Green),Do not lock. Buy at spot.,R$ 0.00,0.00%



--- AI FINANCIAL ROI SUMMARY ---


Month,Cost Without AI (Spot),Cost With AI (Hedge),Saving (R$),Efficiency (%)
2026-03,"R$ 566,665.75","R$ 339,285.10","R$ 227,380.65",40.13
2026-04,"R$ 976,994.05","R$ 976,994.05",R$ 0.00,0.00
2026-05,"R$ 1,456,189.20","R$ 1,456,189.20",R$ 0.00,0.00
2026-06,"R$ 1,380,514.79","R$ 1,380,514.79",R$ 0.00,0.00
